# 03 — Semantic Memory (Recipient Profiles)

Tests `memory/semantic_memory.py`: profile create, update, merge, and retrieval. Uses a temporary file so the real `data/recipient_profiles.json` is never modified.

In [ ]:
import sys, os, json, tempfile
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
from memory.semantic_memory import SemanticMemory

## Helper: temp-file fixture

In [ ]:
def make_mem():
    """Return a SemanticMemory backed by a fresh temp file."""
    tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False, mode='w')
    json.dump({}, tmp)
    tmp.close()
    return SemanticMemory(filepath=tmp.name), tmp.name

## 1. Create a new profile

In [ ]:
mem, path = make_mem()

mem.add_or_update_profile(
    customer_id='cust_001',
    name='wife',
    allergies=['nuts', 'dairy'],
    preferences=['vanilla'],
    location='Colombo'
)

profile = mem.get_profile('cust_001', 'wife')
print('Profile:', json.dumps(profile, indent=2))

assert 'nuts' in profile['allergies']
assert 'dairy' in profile['allergies']
assert 'vanilla' in profile['preferences']
assert profile['location'] == 'Colombo'
print('Create profile: PASSED')

os.unlink(path)

## 2. Update merges allergies & preferences

In [ ]:
mem, path = make_mem()

# First write
mem.add_or_update_profile('cust_001', 'wife', allergies=['nuts'], preferences=['vanilla'], location='Colombo')
# Second write — should merge, not replace
mem.add_or_update_profile('cust_001', 'wife', allergies=['dairy'], preferences=['chocolate'], location='Gampaha')

profile = mem.get_profile('cust_001', 'wife')
print('Merged profile:', json.dumps(profile, indent=2))

assert 'nuts' in profile['allergies'], 'nuts should be preserved after update'
assert 'dairy' in profile['allergies'], 'dairy should be added'
assert 'vanilla' in profile['preferences'], 'vanilla should be preserved'
assert 'chocolate' in profile['preferences'], 'chocolate should be added'
# Location should reflect the latest value
assert profile['location'] == 'Gampaha', f"Expected Gampaha, got {profile['location']}"
print('Update/merge: PASSED')

os.unlink(path)

## 3. Multiple recipients under the same customer

In [ ]:
mem, path = make_mem()

mem.add_or_update_profile('cust_001', 'wife',   allergies=['nuts'],   preferences=['vanilla'],     location='Colombo')
mem.add_or_update_profile('cust_001', 'mother', allergies=['gluten'], preferences=['traditional'], location='Kandy')

wife_profile   = mem.get_profile('cust_001', 'wife')
mother_profile = mem.get_profile('cust_001', 'mother')

print('Wife   profile:', wife_profile)
print('Mother profile:', mother_profile)

assert 'nuts'   in wife_profile['allergies']
assert 'gluten' in mother_profile['allergies']
assert wife_profile['location'] != mother_profile['location']
print('Multiple recipients: PASSED')

os.unlink(path)

## 4. Get profile for unknown customer/recipient

In [ ]:
mem, path = make_mem()

profile = mem.get_profile('nonexistent_customer', 'nobody')
print('Unknown profile:', profile)

# Should return an empty-ish structure without crashing
assert isinstance(profile, dict), 'Should return a dict'
assert profile.get('allergies', []) == [] or profile.get('allergies') is None, \
    'Unknown profile should have no allergies'
print('Unknown profile: PASSED')

os.unlink(path)

## 5. Persistence (save & reload)

In [ ]:
import tempfile

tmp = tempfile.NamedTemporaryFile(suffix='.json', delete=False, mode='w')
json.dump({}, tmp)
tmp.close()
tmp_path = tmp.name

# Write with first instance
mem1 = SemanticMemory(filepath=tmp_path)
mem1.add_or_update_profile('cust_002', 'dad', allergies=['shellfish'], preferences=['whisky'], location='Galle')

# Read with a second instance (simulates a new session)
mem2 = SemanticMemory(filepath=tmp_path)
profile = mem2.get_profile('cust_002', 'dad')
print('Reloaded profile:', profile)

assert 'shellfish' in profile['allergies']
assert 'whisky' in profile['preferences']
assert profile['location'] == 'Galle'
print('Persistence test: PASSED')

os.unlink(tmp_path)

## 6. Deduplication of allergies

In [ ]:
mem, path = make_mem()

# Add the same allergy twice across two calls
mem.add_or_update_profile('cust_003', 'brother', allergies=['nuts'], preferences=[], location='')
mem.add_or_update_profile('cust_003', 'brother', allergies=['nuts'], preferences=[], location='')

profile = mem.get_profile('cust_003', 'brother')
print('Allergies after duplicate adds:', profile['allergies'])

# Ideally deduplicated; at minimum nuts appears
assert 'nuts' in profile['allergies']
assert profile['allergies'].count('nuts') == 1, f"Expected 1 'nuts', got: {profile['allergies']}"
print('Deduplication test: PASSED')

os.unlink(path)